In [1]:
!wget https://storage.yandexcloud.net/ai-2025/audio.zip

--2026-04-18 13:19:14--  https://storage.yandexcloud.net/ai-2025/audio.zip
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10863973 (10M) [application/zip]
Saving to: ‘audio.zip’

audio.zip           100%[===================>]  10.36M  3.90MB/s    in 2.7s    

2026-04-18 13:19:19 (3.90 MB/s) - ‘audio.zip’ saved [10863973/10863973]



In [2]:
!unzip audio.zip

Archive:  audio.zip
  inflating: 5.wav                   
  inflating: 4.wav                   
  inflating: 1.wav                   
  inflating: 3.wav                   
  inflating: 2.wav                   


In [3]:
!pip install SpeechRecognition
!pip install librosa soundfile numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 69.5 MB/s eta 0:00:00


In [4]:
import speech_recognition as sr
import librosa
import soundfile as sf
import numpy as np
import os

In [5]:
# Полные тексты стихотворений
reference_texts = {
    '1.wav': """Тигр Тигр жгучий страх ты горишь в ночных лесах
Чей бессмертный взор любя создал страшного тебя
В небесах и средь дубрав чьей рукой разлит твой нрав
чьей расплавлен был рукой огонь разлитый тобой""",

    '2.wav': """Я помню чудное мгновенье передо мной явилась ты
как мимолетное виденье как гений чистой красоты
В томленьях грусти безнадежной в тревогах шумной суеты
звучал мне долго голос нежный и снились милые черты""",

    '3.wav': """Мне голос был он звал утешно он говорил иди сюда
оставь свой край глухой и грешный оставь Россию навсегда
Но равнодушно и спокойно руками я замкнула слух
чтоб этой речью недостойной не осквернился скорбный дух""",

    '4.wav': """Выхожу один я на дорогу сквозь туман кремнистый путь блестит
Ночь тиха пустыня внемлет богу и звезда с звездою говорит
В небесах торжественно и чудно спит земля в сиянье голубом
что же мне так больно и так трудно жду ль чего жалею ли о чем""",

    '5.wav': """Хозяин здесь не в первый раз он не застанет к счастью нас
остановлюсь на полпути лошадке странно ни души
В лесу темно а лес глубок и долг велит держать мне срок
но прежде чем в седле заснуть еще не близок мой ночлег"""
}

In [6]:
def add_noise(audio_path, noise_level=0.005):
    """Добавляет белый шум к аудиофайлу"""
    audio, sr_rate = librosa.load(audio_path, sr=None)
    noise = np.random.normal(0, noise_level, audio.shape)
    noisy_audio = audio + noise
    noisy_path = audio_path.replace('.wav', f'_noisy_{noise_level}.wav')
    sf.write(noisy_path, noisy_audio, sr_rate)
    return noisy_path

def recognize_speech(audio_path):
    """Распознает речь из аудиофайла через Google API"""
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)
    try:
        text = recognizer.recognize_google(audio, language='ru-RU')
        return text
    except sr.UnknownValueError:
        return "[НЕ РАСПОЗНАНО]"
    except sr.RequestError as e:
        return f"[ОШИБКА: {e}]"

def calculate_metrics(reference, recognized):
    """
    Вычисляет метрики качества распознавания:
    1. Word Accuracy (точность по словам)
    2. Jaccard Similarity (коэффициент Жаккара)
    3. Character Accuracy (по символам)
    """
    if recognized in ["[НЕ РАСПОЗНАНО]", "[ОШИБКА:"] or not recognized:
        return 0.0, 0.0, 0.0

    ref_words = reference.lower().split()
    rec_words = recognized.lower().split()

    # Метрика 1: Word Accuracy (сколько слов из эталона распознано)
    correct_words = sum(1 for rw in rec_words if rw in ref_words)
    word_acc = correct_words / len(ref_words) if ref_words else 0

    # Метрика 2: Jaccard (схожесть множеств)
    set_ref = set(ref_words)
    set_rec = set(rec_words)
    intersection = len(set_ref & set_rec)
    union = len(set_ref | set_rec)
    jaccard = intersection / union if union > 0 else 0

    # Метрика 3: Character Accuracy (простое совпадение строк)
    # Очищаем от знаков препинания для простоты
    import re
    ref_clean = re.sub(r'[^\w\s]', '', reference.lower())
    rec_clean = re.sub(r'[^\w\s]', '', recognized.lower())

    # Подсчет совпадающих символов
    min_len = min(len(ref_clean), len(rec_clean))
    matches = sum(1 for i in range(min_len) if ref_clean[i] == rec_clean[i])
    char_acc = matches / max(len(ref_clean), 1)

    return word_acc, jaccard, char_acc

In [7]:
# Список ваших аудиофайлов
audio_files = ['1.wav', '2.wav', '3.wav', '4.wav', '5.wav']

# Разные уровни шума для тестирования
noise_levels = [0, 0.001, 0.003, 0.005, 0.01]

results = []

print("="*100)
print("ЭКСПЕРИМЕНТ: ВЛИЯНИЕ ШУМА НА ТОЧНОСТЬ РАСПОЗНАВАНИЯ РЕЧИ")
print("="*100)

for audio_file in audio_files:
    print(f"\n\n{'='*100}")
    print(f" АУДИОФАЙЛ: {audio_file}")
    print(f"{'='*100}")

    for noise in noise_levels:
        if noise == 0:
            audio_path = audio_file
            noise_desc = " БЕЗ ШУМА"
        else:
            audio_path = add_noise(audio_file, noise)
            noise_desc = f" ШУМ (ур. {noise})"

        # Распознавание
        recognized = recognize_speech(audio_path)

        # Вычисление метрик
        ref_text = reference_texts[audio_file]
        word_acc, jaccard, char_acc = calculate_metrics(ref_text, recognized)

        # Сохранение результатов
        results.append({
            'Файл': audio_file,
            'Уровень_шума': noise,
            'Шум': noise_desc,
            'Word_Accuracy': round(word_acc, 3),
            'Jaccard': round(jaccard, 3),
            'Char_Accuracy': round(char_acc, 3),
            'Распознанный_текст': recognized[:150] + "..." if len(recognized) > 150 else recognized
        })

        # Вывод
        print(f"\n  {noise_desc}")
        print(f"   Точность по словам: {word_acc:.1%}")
        print(f"   Коэф. Жаккара: {jaccard:.1%}")
        print(f"   Точность по символам: {char_acc:.1%}")
        print(f"   Распознано: {recognized[:100]}...")

ЭКСПЕРИМЕНТ: ВЛИЯНИЕ ШУМА НА ТОЧНОСТЬ РАСПОЗНАВАНИЯ РЕЧИ


 АУДИОФАЙЛ: 1.wav

   БЕЗ ШУМА
   Точность по словам: 81.8%
   Коэф. Жаккара: 79.3%
   Точность по символам: 61.5%
   Распознано: Тигр тигр жгучий страх ты горишь в ночных лесах Чей бессмертный взор любя создал страшного тебя в не...

   ШУМ (ур. 0.001)
   Точность по словам: 81.8%
   Коэф. Жаккара: 79.3%
   Точность по символам: 61.5%
   Распознано: Тигр тигр жгучий страх ты горишь в ночных лесах Чей бессмертный взор любя создал страшного тебя в не...

   ШУМ (ур. 0.003)
   Точность по словам: 81.8%
   Коэф. Жаккара: 79.3%
   Точность по символам: 61.5%
   Распознано: Тигр тигр жгучий страх ты горишь в ночных лесах Чей бессмертный взор любя создал страшного тебя в не...

   ШУМ (ур. 0.005)
   Точность по словам: 81.8%
   Коэф. Жаккара: 79.3%
   Точность по символам: 61.5%
   Распознано: Тигр тигр жгучий страх ты горишь в ночных лесах Чей бессмертный взор любя создал страшного тебя в не...

   ШУМ (ур. 0.01)
   Точность по слов

In [8]:
print("="*80)
print("АНАЛИЗ РЕЗУЛЬТАТОВ ПО ФАЙЛАМ")
print("="*80)

analysis = {
    '1.wav (Блейк)': {
        'точность': '81.8%',
        'вердикт': 'Хорошо',
        'причина': 'Специфическая поэтическая лексика ("жгучий", "разлит")'
    },
    '2.wav (Пушкин)': {
        'точность': '96.9%',
        'вердикт': 'Отлично',
        'причина': 'Классическая, хорошо узнаваемая лексика, четкая дикция'
    },
    '3.wav (Ахматова)': {
        'точность': '97.1%',
        'вердикт': 'Отлично',
        'причина': 'Четкое произношение, хорошая громкость'
    },
    '4.wav (Фрост)': {
        'точность': '15.6%',
        'вердикт': 'Плохо',
        'причина': 'Возможные проблемы с записью: тихая речь, акцент, фоновый шум'
    },
    '5.wav (Лермонтов)': {
        'точность': '16.7%',
        'вердикт': 'Плохо',
        'причина': 'Аналогично файлу 4 - технические проблемы записи'
    }
}

for file, data in analysis.items():
    print(f"\n {file}")
    print(f"   Точность: {data['точность']} → {data['вердикт']}")
    print(f"    {data['причина']}")

АНАЛИЗ РЕЗУЛЬТАТОВ ПО ФАЙЛАМ

 1.wav (Блейк)
   Точность: 81.8% → Хорошо
    Специфическая поэтическая лексика ("жгучий", "разлит")

 2.wav (Пушкин)
   Точность: 96.9% → Отлично
    Классическая, хорошо узнаваемая лексика, четкая дикция

 3.wav (Ахматова)
   Точность: 97.1% → Отлично
    Четкое произношение, хорошая громкость

 4.wav (Фрост)
   Точность: 15.6% → Плохо
    Возможные проблемы с записью: тихая речь, акцент, фоновый шум

 5.wav (Лермонтов)
   Точность: 16.7% → Плохо
    Аналогично файлу 4 - технические проблемы записи


**ВЫВОДЫ:**
Проведен эксперимент по распознаванию речи с добавлением белого шума
на 5 аудиозаписях поэтических произведений (Блейк, Пушкин, Ахматова,
Роберт Фрост, Лермонтов).

РЕЗУЛЬТАТЫ:
1. Шум в диапазоне [0, 0.01] НЕ оказал значимого влияния на точность
   распознавания благодаря встроенным алгоритмам Google Speech API.

2. Точность распознавания варьируется от 15.6% до 97.1%:
   - Лучшие результаты: Пушкин (96.9%) и Ахматова (97.1%)
   - Худшие результаты: Фрост (15.6%) и Лермонтов (16.7%)

3. КЛЮЧЕВОЙ ВЫВОД: качество распознавания определяется не шумом,
   а исходным качеством записи (громкость, четкость дикции, темп речи).

РЕКОМЕНДАЦИИ ДЛЯ ПОВЫШЕНИЯ ТОЧНОСТИ:

• Записывать аудио с одинаковой громкостью для всех образцов

• Говорить четко, размеренно, с паузами между строками

• Использовать качественный микрофон в тихом помещении